<a href="https://colab.research.google.com/github/Delean-Mafra/faculdade/blob/main/Aplicacoes_da_Ciencia_de_Dadosem_Tecnologias_Disruptivas/GOTURN_(Generic_Object_Tracking_Using_Regression_Networks).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# ============================================================
# Rastreamento: GOTURN + CSRT + KCF + Kalman + HOG + IoU + HSV
# Google Colab | Estados exclusivos: CONFIRMADO, PREVISTO, PERDIDO
# ============================================================

import os
import math
import time
import base64
import urllib.request
import subprocess
import zipfile
import argparse
import unittest
from dataclasses import dataclass
from collections import deque

import cv2
import numpy as np
from IPython.display import HTML, display

# ------------------------- 1. CONFIGURACAO -------------------------
VIDEO_IN = '/content/rastreamento_saida.mp4'
VIDEO_OUT = '/content/resultado_avancado_h264.mp4'
TEMP_OUT = '/content/temp_out.mp4'
BBOX_MANUAL = (100, 100, 150, 200)
ZONA_PROMO_REL = (0.50, 0.30, 0.40, 0.60)

IOU_MINIMO_CONSENSO = 0.40
BHATTACHARYYA_MAX_REFERENCIA = 0.45
BHATTACHARYYA_MAX_ADAPTATIVA = 0.38
HSV_UPDATE_MAX_DIST = 0.30
HSV_ADAPT_ALPHA = 0.08
HSV_UPDATE_INTERVAL_S = 0.60
HOG_AUTO_MIN_SCORE = 0.60
HOG_AUTO_MIN_MARGIN = 0.12
SELECAO_MANUAL_SE_AMBIGUA = True
# No Colab, ajuste BBOX_MANUAL visualmente e altere para True antes de criar hist_ref.
BBOX_MANUAL_CONFIRMADA = False
KALMAN_VEL_MAX_DIAG_S = 0.20
KALMAN_VEL_DAMPING_PERDIDO = 0.82
KALMAN_MAX_DRIFT_DIAG_REL = 0.30
KALMAN_SIGMA_BUSCA = 2.5
DIST_REDETECCAO_MAX_DIAG_REL = 0.45
AREA_RATIO_MIN = 0.50
AREA_RATIO_MAX = 2.00

# Parametros temporais/espaciais realmente ajustados depois da leitura do video.
OCLUSAO_MAX_S = 0.60
DIST_TRACKER_DIAG_REL = 0.12
DIST_REDETECCAO_DIAG_REL = 0.25
KALMAN_ACCEL_NOISE_REL = 0.0025
KALMAN_MEAS_NOISE_REL = 0.0030

MODEL_DIR = '/content'
MODEL_TXT = f'{MODEL_DIR}/goturn.prototxt'
MODEL_BIN = f'{MODEL_DIR}/goturn.caffemodel'
GOTURN_BASE_URL = (
    'https://raw.githubusercontent.com/opencv/opencv_extra/'
    'c4219d5eb3105ed8e634278fad312a1a8d2c182d/testdata/tracking'
)

ESTADOS_VALIDOS = {'CONFIRMADO', 'PREVISTO', 'PERDIDO'}
ORIGENS_VALIDAS = {'HOG_INICIAL', 'MANUAL', 'ENSEMBLE', 'HOG_HSV', 'HOG_SEM_HSV', 'KALMAN', 'NENHUMA'}


@dataclass
class ParametrosDinamicos:
    fps: float
    largura: int
    altura: int
    diagonal: float
    dt: float
    max_frames_oclusao: int
    max_dist_tracker: float
    max_dist_redeteccao: float
    process_noise: float
    measurement_noise: float
    hsv_update_interval_frames: int


@dataclass
class ResultadoRedeteccao:
    bbox: tuple | None
    confianca: str                 # ALTA, REDUZIDA ou NENHUMA
    origem: str                    # HOG_HSV, HOG_SEM_HSV ou NENHUMA
    distancia_hsv: float | None


def calcular_parametros_dinamicos(fps, largura, altura):
    fps = float(fps) if fps and fps > 1e-3 else 25.0
    diagonal = math.hypot(largura, altura)
    dt = 1.0 / fps
    # Limites temporais convertidos para frames, e distancias escaladas pela diagonal.
    max_frames_oclusao = max(1, int(round(OCLUSAO_MAX_S * fps)))
    max_dist_tracker = max(20.0, DIST_TRACKER_DIAG_REL * diagonal)
    max_dist_redeteccao = max(40.0, DIST_REDETECCAO_DIAG_REL * diagonal)
    # Covariancias em pixels, escaladas pela resolucao e pelo passo temporal.
    sigma_acc = max(1.0, KALMAN_ACCEL_NOISE_REL * diagonal)
    sigma_med = max(1.0, KALMAN_MEAS_NOISE_REL * diagonal)
    process_noise = sigma_acc ** 2
    measurement_noise = sigma_med ** 2
    hsv_interval = max(1, int(round(HSV_UPDATE_INTERVAL_S * fps)))
    return ParametrosDinamicos(
        fps, largura, altura, diagonal, dt, max_frames_oclusao,
        max_dist_tracker, max_dist_redeteccao,
        process_noise, measurement_noise, hsv_interval
    )


# ------------------------- 2. GOTURN -------------------------
def baixar_modelo_goturn():
    os.makedirs(MODEL_DIR, exist_ok=True)
    try:
        if not os.path.exists(MODEL_TXT) or os.path.getsize(MODEL_TXT) == 0:
            print('Baixando goturn.prototxt...')
            urllib.request.urlretrieve(f'{GOTURN_BASE_URL}/goturn.prototxt', MODEL_TXT)

        if not os.path.exists(MODEL_BIN) or os.path.getsize(MODEL_BIN) < 1_000_000:
            print('Baixando e montando pesos do GOTURN...')
            partes = [f'{MODEL_DIR}/goturn.caffemodel.zip.00{i}' for i in range(1, 5)]
            for i, parte in enumerate(partes, 1):
                if not os.path.exists(parte) or os.path.getsize(parte) == 0:
                    urllib.request.urlretrieve(
                        f'{GOTURN_BASE_URL}/goturn.caffemodel.zip.00{i}', parte
                    )
            zip_path = f'{MODEL_DIR}/goturn.caffemodel.zip'
            with open(zip_path, 'wb') as saida:
                for parte in partes:
                    with open(parte, 'rb') as entrada:
                        saida.write(entrada.read())
            with zipfile.ZipFile(zip_path, 'r') as arquivo_zip:
                arquivo_zip.extractall(MODEL_DIR)

        if not os.path.exists(MODEL_TXT) or not os.path.exists(MODEL_BIN):
            raise FileNotFoundError('Arquivos do modelo GOTURN nao foram encontrados apos o download.')
        print('Modelo GOTURN disponivel.')
        return True
    except Exception as exc:
        print(f'[ERRO GOTURN] Nao foi possivel obter/carregar os arquivos do modelo: {exc}')
        return False


# ------------------------- 3. KALMAN -------------------------
def criar_kalman(params):
    kf = cv2.KalmanFilter(4, 2)
    dt = np.float32(params.dt)
    kf.transitionMatrix = np.array(
        [[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]],
        np.float32
    )
    kf.measurementMatrix = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], np.float32)

    # Modelo de aceleracao branca discreta, dependente de dt e resolucao.
    q = np.float32(params.process_noise)
    dt2, dt3, dt4 = dt * dt, dt * dt * dt, dt * dt * dt * dt
    kf.processNoiseCov = q * np.array(
        [[dt4/4, 0, dt3/2, 0], [0, dt4/4, 0, dt3/2],
         [dt3/2, 0, dt2, 0], [0, dt3/2, 0, dt2]], np.float32
    )
    kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * np.float32(params.measurement_noise)
    kf.errorCovPost = np.eye(4, dtype=np.float32) * np.float32(params.measurement_noise)
    return kf


def inicializar_kalman(kf, cx, cy, params=None):
    """Reinicia estado e incerteza, inclusive depois de uma recuperacao."""
    estado = np.array([[cx], [cy], [0], [0]], np.float32)
    kf.statePost = estado.copy()
    kf.statePre = estado.copy()
    if params is not None:
        var_pos = max(1.0, float(params.measurement_noise))
    else:
        var_pos = max(1.0, float(np.mean(np.diag(kf.measurementNoiseCov))))
    # Velocidade nasce menos conhecida que a posicao medida, mas sem herdar
    # a incerteza acumulada na oclusao anterior.
    cov_inicial = np.diag([var_pos, var_pos, 4.0*var_pos, 4.0*var_pos]).astype(np.float32)
    kf.errorCovPost = cov_inicial.copy()
    kf.errorCovPre = cov_inicial.copy()


def propagar_sem_medicao(kf):
    """Promove integralmente a priori para a posteriori quando nao ha medicao.

    O OpenCV calcula statePre/errorCovPre em predict(), mas a proxima previsao
    deve partir explicitamente desses valores quando correct() nao e chamado.
    """
    kf.statePost = kf.statePre.copy()
    kf.errorCovPost = kf.errorCovPre.copy()
    return kf.statePost.copy(), kf.errorCovPost.copy()


def limitar_predicao_perdida(kf, pred, ancora, params, largura, altura):
    """Restringe a priori e a promove integralmente para a posteriori."""
    estado = np.asarray(pred, dtype=np.float32).copy()
    vx = float(estado[2, 0]) * KALMAN_VEL_DAMPING_PERDIDO
    vy = float(estado[3, 0]) * KALMAN_VEL_DAMPING_PERDIDO
    vmax = KALMAN_VEL_MAX_DIAG_S * params.diagonal
    velocidade = math.hypot(vx, vy)
    if velocidade > vmax and velocidade > 0:
        escala = vmax / velocidade
        vx, vy = vx * escala, vy * escala

    px, py = float(estado[0, 0]), float(estado[1, 0])
    ax, ay = ancora
    limite_deriva = KALMAN_MAX_DRIFT_DIAG_REL * params.diagonal
    dx, dy = px - ax, py - ay
    distancia = math.hypot(dx, dy)
    if distancia > limite_deriva and distancia > 0:
        escala = limite_deriva / distancia
        px, py = ax + dx * escala, ay + dy * escala
    px = float(np.clip(px, 0, largura - 1))
    py = float(np.clip(py, 0, altura - 1))

    estado[:, 0] = (px, py, vx, vy)
    # Promocao completa: posicao, velocidade e covariancia previstas tornam-se
    # a posteriori do frame corrente. Assim o proximo predict() realmente avanca.
    kf.statePost = estado.copy()
    kf.statePre = estado.copy()
    kf.errorCovPost = kf.errorCovPre.copy()

    cov_xy = np.asarray(kf.errorCovPost[:2, :2], dtype=float)
    sigma = math.sqrt(max(0.0, float(np.trace(cov_xy))))
    raio = params.max_dist_redeteccao + KALMAN_SIGMA_BUSCA * sigma
    raio_max = DIST_REDETECCAO_MAX_DIAG_REL * params.diagonal
    return px, py, min(raio, raio_max), sigma


def exibir_previa_bbox_manual(frame, bbox):
    """Mostra no Colab a ROI manual que deve ser conferida antes da confirmacao."""
    previa = frame.copy()
    x, y, w, h = limitar_bbox(bbox, frame.shape[1], frame.shape[0])
    cv2.rectangle(previa, (x, y), (x+w, y+h), (0, 255, 255), 3)
    cv2.putText(previa, 'CONFIRA BBOX_MANUAL', (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    ok, buffer = cv2.imencode('.jpg', previa)
    if ok:
        imagem64 = base64.b64encode(buffer).decode()
        display(HTML(f'<img style="max-width:800px" src="data:image/jpeg;base64,{imagem64}">'))
    print(f'[BBOX_MANUAL] Atual: {(x, y, w, h)}. Ajuste BBOX_MANUAL e defina BBOX_MANUAL_CONFIRMADA=True para criar hist_ref.')


def preparar_selecao_manual(video_path=VIDEO_IN, bbox=BBOX_MANUAL):
    """Etapa 1 do Colab: exibe o primeiro frame e a caixa, sem iniciar trackers."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f'Nao foi possivel abrir o video para conferencia: {video_path}')
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError('Nao foi possivel ler o primeiro frame para conferencia manual.')
    bbox_ajustada = limitar_bbox(bbox, frame.shape[1], frame.shape[0])
    exibir_previa_bbox_manual(frame, bbox_ajustada)
    return bbox_ajustada


def identidade_manual_liberada(origem, manual_confirmada, hist_ref=None):
    """Confirmacao humana e disponibilidade HSV sao informacoes independentes."""
    return origem != 'MANUAL' or bool(manual_confirmada)


def estabelecer_assinatura_inicial(frame, bbox, origem, manual_confirmada=False):
    """Impede que BBOX manual nao conferida se torne identidade permanente."""
    if origem == 'MANUAL' and not manual_confirmada:
        print('[HSV] BBOX_MANUAL ainda nao confirmada; histograma de referencia nao foi criado.')
        return None
    return extrair_histograma(frame, bbox)


def construir_texto_hud(estado, origem, modo_consenso, votos, ativos, maioria,
                         frames_sem, max_oclusao, tem_hsv, recuperado_neste_frame=False,
                         raio_busca=None, sigma=None):
    if recuperado_neste_frame:
        confianca = 'ALTA' if origem == 'HOG_HSV' else 'REDUZIDA'
        return (f'RECUPERADO {origem} confianca={confianca} | '
                f'trackers_reinicializados={ativos} | aguardando votos')
    if estado == 'CONFIRMADO':
        rotulo = 'TRACKER_UNICO confianca=REDUZIDA' if modo_consenso == 'TRACKER_UNICO' else 'ENSEMBLE'
        return (f'CONFIRMADO {rotulo} origem={origem} | '
                f'votos={votos}/{ativos} | maioria={maioria}')
    if estado == 'PREVISTO':
        return (f'PREVISTO Kalman | sem consenso={frames_sem}/{max_oclusao} | '
                f'votos={votos}/{ativos}')
    modo = 'HOG+HSV' if tem_hsv else 'HOG geometria (baixa confianca)'
    extra = '' if raio_busca is None else f' | raio={raio_busca:.0f}px sigma={sigma:.1f}px'
    return f'PERDIDO | buscando={modo} | ativos={ativos}{extra}'


# ------------------------- 4. TRACKERS -------------------------
def _criar_tracker_csrt():
    if hasattr(cv2, 'TrackerCSRT_create'):
        return cv2.TrackerCSRT_create()
    if hasattr(cv2, 'legacy') and hasattr(cv2.legacy, 'TrackerCSRT_create'):
        return cv2.legacy.TrackerCSRT_create()
    raise RuntimeError('CSRT indisponivel. Instale opencv-contrib-python.')


def _criar_tracker_kcf():
    if hasattr(cv2, 'TrackerKCF_create'):
        return cv2.TrackerKCF_create()
    if hasattr(cv2, 'legacy') and hasattr(cv2.legacy, 'TrackerKCF_create'):
        return cv2.legacy.TrackerKCF_create()
    raise RuntimeError('KCF indisponivel. Instale opencv-contrib-python.')


def _criar_tracker_goturn():
    if not os.path.exists(MODEL_TXT) or not os.path.exists(MODEL_BIN):
        raise RuntimeError('Modelos GOTURN ausentes.')

    namespaces = [('cv2', cv2)]
    if hasattr(cv2, 'legacy'):
        namespaces.append(('cv2.legacy', cv2.legacy))
    erros = []
    for nome_api, api in namespaces:
        criar = getattr(api, 'TrackerGOTURN_create', None)
        params_cls = getattr(api, 'TrackerGOTURN_Params', None)
        if criar is None:
            erros.append(f'{nome_api}: fabrica ausente')
            continue
        try:
            if params_cls is not None:
                params = params_cls()
                params.modelTxt = MODEL_TXT
                params.modelBin = MODEL_BIN
                return criar(params)
            # Sem Params nao ha como garantir que MODEL_TXT/MODEL_BIN em /content
            # sejam usados. Nao cria um GOTURN potencialmente ligado a modelos errados.
            erros.append(f'{nome_api}: fabrica sem TrackerGOTURN_Params; modelos personalizados nao podem ser garantidos')
            continue
        except Exception as exc:
            erros.append(f'{nome_api}: {type(exc).__name__}: {exc}')
    raise RuntimeError('GOTURN nao inicializado. ' + ' | '.join(erros))


def inicializar_trackers(frame, bbox, habilitar_goturn=True):
    fabricas = {'CSRT': _criar_tracker_csrt, 'KCF': _criar_tracker_kcf}
    if habilitar_goturn:
        fabricas['GOTURN'] = _criar_tracker_goturn

    candidatos = {}
    for nome, fabrica in fabricas.items():
        try:
            candidatos[nome] = fabrica()
        except Exception as exc:
            print(f'[INDISPONIVEL] {nome}: {exc}')

    ativos = {}
    bbox = tuple(float(v) for v in bbox)
    for nome, tracker in candidatos.items():
        try:
            retorno = tracker.init(frame, bbox)
            # Em algumas versoes init retorna None em caso de sucesso.
            ok = True if retorno is None else bool(retorno)
            if ok:
                ativos[nome] = tracker
            else:
                print(f'[FALHA INIT] {nome}: init retornou falso.')
        except Exception as exc:
            print(f'[FALHA INIT] {nome}: {type(exc).__name__}: {exc}')

    print(f'Rastreadores ativos ({len(ativos)}): {list(ativos.keys())}')
    return ativos


# ------------------------- 5. APARENCIA E GEOMETRIA -------------------------
def limitar_bbox(bbox, largura, altura):
    x, y, w, h = [int(round(v)) for v in bbox]
    x = max(0, min(x, largura - 1))
    y = max(0, min(y, altura - 1))
    w = max(1, min(w, largura - x))
    h = max(1, min(h, altura - y))
    return (x, y, w, h)


def extrair_histograma(frame, bbox):
    h_frame, w_frame = frame.shape[:2]
    x, y, w, h = limitar_bbox(bbox, w_frame, h_frame)
    roi = frame[y:y+h, x:x+w]
    if roi.size == 0:
        return None
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    # Mascara reduz influencia de pixels quase cinzentos/escuros.
    mascara = cv2.inRange(hsv, np.array((0, 30, 20)), np.array((180, 255, 255)))
    hist = cv2.calcHist([hsv], [0, 1], mascara, [16, 16], [0, 180, 0, 256])
    if hist is None or float(hist.sum()) <= 0:
        return None
    cv2.normalize(hist, hist, alpha=1.0, norm_type=cv2.NORM_L1)
    return hist.astype(np.float32)


def distancia_hist(hist_a, hist_b):
    if hist_a is None or hist_b is None:
        return None
    return float(cv2.compareHist(hist_a, hist_b, cv2.HISTCMP_BHATTACHARYYA))


def atualizar_assinatura_controlada(frame, bbox, hist_ref, hist_adapt):
    candidato = extrair_histograma(frame, bbox)
    if candidato is None or hist_ref is None:
        return hist_adapt, False, None
    d_ref = distancia_hist(hist_ref, candidato)
    d_adapt = distancia_hist(hist_adapt, candidato) if hist_adapt is not None else d_ref
    compativel = d_ref <= HSV_UPDATE_MAX_DIST and d_adapt <= HSV_UPDATE_MAX_DIST
    if not compativel:
        return hist_adapt, False, d_ref
    if hist_adapt is None:
        novo = candidato.copy()
    else:
        novo = (1.0 - HSV_ADAPT_ALPHA) * hist_adapt + HSV_ADAPT_ALPHA * candidato
        cv2.normalize(novo, novo, alpha=1.0, norm_type=cv2.NORM_L1)
    return novo.astype(np.float32), True, d_ref


def calcular_iou(a, b):
    xa, ya = max(a[0], b[0]), max(a[1], b[1])
    xb, yb = min(a[0]+a[2], b[0]+b[2]), min(a[1]+a[3], b[1]+b[3])
    inter = max(0.0, xb-xa) * max(0.0, yb-ya)
    if inter <= 0:
        return 0.0
    area_a, area_b = max(0.0, a[2]*a[3]), max(0.0, b[2]*b[3])
    uniao = area_a + area_b - inter
    return inter / uniao if uniao > 0 else 0.0


def _grupo_compativel_todos_pares(indices, caixas):
    return all(
        calcular_iou(caixas[i], caixas[j]) >= IOU_MINIMO_CONSENSO
        for pos, i in enumerate(indices) for j in indices[pos+1:]
    )


def obter_consenso_ensemble(caixas_validas, total_trackers_ativos):
    """Maior clique de IoU: toda dupla do grupo deve satisfazer o limiar."""
    from itertools import combinations
    maioria = total_trackers_ativos // 2 + 1 if total_trackers_ativos > 0 else 0
    modo = 'TRACKER_UNICO' if total_trackers_ativos == 1 else 'ENSEMBLE'
    if maioria == 0 or len(caixas_validas) < maioria:
        return None, 0, maioria, modo

    caixas = list(caixas_validas.values())
    melhor = ()
    # Busca exata e deterministica. O ensemble tem no maximo tres algoritmos aqui.
    for tamanho in range(len(caixas), maioria - 1, -1):
        candidatos = []
        for grupo in combinations(range(len(caixas)), tamanho):
            if _grupo_compativel_todos_pares(grupo, caixas):
                candidatos.append(grupo)
        if candidatos:
            # Desempate: maior IoU medio entre os pares.
            def iou_medio(grupo):
                pares = [calcular_iou(caixas[i], caixas[j])
                         for pos, i in enumerate(grupo) for j in grupo[pos+1:]]
                return sum(pares)/len(pares) if pares else 1.0
            melhor = max(candidatos, key=iou_medio)
            break
    if not melhor:
        return None, 0, maioria, modo
    box = tuple(np.mean(np.asarray([caixas[i] for i in melhor], dtype=np.float32), axis=0).tolist())
    return box, len(melhor), maioria, modo


hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())


def selecionar_alvo_inicial(frame, bbox_manual=BBOX_MANUAL):
    caixas, pesos = hog.detectMultiScale(frame, winStride=(8, 8), padding=(8, 8), scale=1.05)
    candidatos = []
    pesos_flat = np.asarray(pesos, dtype=float).reshape(-1) if len(pesos) else np.array([])
    for i, (x, y, w, h) in enumerate(caixas):
        score = float(pesos_flat[i]) if i < len(pesos_flat) else 0.0
        box = (int(x+w*0.1), int(y+h*0.1), int(w*0.8), int(h*0.8))
        candidatos.append((score, box))
    candidatos.sort(key=lambda item: item[0], reverse=True)
    if not candidatos:
        return limitar_bbox(bbox_manual, frame.shape[1], frame.shape[0]), 'MANUAL', True, []
    melhor_score, melhor_box = candidatos[0]
    segundo = candidatos[1][0] if len(candidatos) > 1 else -float('inf')
    ambiguo = (melhor_score < HOG_AUTO_MIN_SCORE or
               (len(candidatos) > 1 and melhor_score - segundo < HOG_AUTO_MIN_MARGIN))
    if ambiguo and SELECAO_MANUAL_SE_AMBIGUA:
        print(f'[SELECAO] HOG ambiguo: melhor={melhor_score:.3f}, segundo={segundo:.3f}. Usando BBOX_MANUAL.')
        return limitar_bbox(bbox_manual, frame.shape[1], frame.shape[0]), 'MANUAL', True, candidatos
    return melhor_box, 'HOG_INICIAL', False, candidatos


def tentar_redeteccao(frame, kx, ky, max_dist, area_esperada=None,
                      hist_ref=None, hist_adapt=None, inicial=False):
    caixas, pesos = hog.detectMultiScale(
        frame, winStride=(8, 8), padding=(8, 8), scale=1.05
    )
    melhor = None
    melhor_score = float('inf')
    melhor_bhat = None
    origem = 'NENHUMA'

    for (x, y, w, h) in caixas:
        box = (int(x+w*0.1), int(y+h*0.1), int(w*0.8), int(h*0.8))
        cx, cy = box[0]+box[2]/2, box[1]+box[3]/2
        dist = math.hypot(cx-kx, cy-ky)
        if dist > max_dist:
            continue
        if not inicial:
            if not area_esperada or area_esperada <= 0:
                continue
            ratio = (box[2]*box[3]) / float(area_esperada)
            if not (AREA_RATIO_MIN <= ratio <= AREA_RATIO_MAX):
                continue

        if inicial:
            score, bhat, cand_origem = dist, None, 'HOG_INICIAL'
        else:
            hist_cand = extrair_histograma(frame, box)
            # Referencia imutavel tem prioridade. Adaptativa ajuda, mas nao substitui a original.
            if hist_ref is not None and hist_cand is not None:
                d_ref = distancia_hist(hist_ref, hist_cand)
                d_adapt = distancia_hist(hist_adapt, hist_cand) if hist_adapt is not None else d_ref
                if d_ref > BHATTACHARYYA_MAX_REFERENCIA or d_adapt > BHATTACHARYYA_MAX_ADAPTATIVA:
                    continue
                bhat = max(d_ref, d_adapt)
                score = dist/max_dist + bhat
                cand_origem = 'HOG_HSV'
            elif hist_ref is None:
                # Ausencia explicita de HSV: usa geometria+distancia e marca confianca reduzida.
                bhat = None
                score = dist/max_dist
                cand_origem = 'HOG_SEM_HSV'
            else:
                # Existe referencia, mas candidato nao produziu histograma: rejeitar.
                continue

        if score < melhor_score:
            melhor, melhor_score, melhor_bhat, origem = box, score, bhat, cand_origem

    if melhor is None:
        return ResultadoRedeteccao(None, 'NENHUMA', 'NENHUMA', None)
    confianca = 'ALTA' if origem in {'HOG_INICIAL', 'HOG_HSV'} else 'REDUZIDA'
    return ResultadoRedeteccao(melhor, confianca, origem, melhor_bhat)


# ------------------------- 6. ESTADOS E METRICAS -------------------------
def decidir_estado(estado_anterior, houve_consenso, frames_sem_consenso, max_frames_oclusao):
    if houve_consenso:
        return 'CONFIRMADO', 0
    novos_frames = frames_sem_consenso + 1
    if novos_frames <= max_frames_oclusao:
        return 'PREVISTO', novos_frames
    return 'PERDIDO', novos_frames


def contabilizar_estado(metricas, estado):
    if estado not in ESTADOS_VALIDOS:
        raise ValueError(f'Estado invalido: {estado}')
    metricas['total_frames'] += 1
    mapa = {'CONFIRMADO': 'confirmados', 'PREVISTO': 'previstos', 'PERDIDO': 'perdidos'}
    metricas[mapa[estado]] += 1


def validar_metricas(metricas):
    soma = metricas['confirmados'] + metricas['previstos'] + metricas['perdidos']
    if soma != metricas['total_frames']:
        raise AssertionError(f'Metricas inconsistentes: categorias={soma}, total={metricas["total_frames"]}')


# ------------------------- 7. PROCESSAMENTO -------------------------
def passos_previsao_esperados(total_frames):
    """Primeiro frame inicializa; cada frame restante consome um predict()."""
    return max(0, int(total_frames) - 1)


def processar_video():
    goturn_ok = baixar_modelo_goturn()
    if not os.path.exists(VIDEO_IN):
        raise FileNotFoundError(f'Video de entrada nao encontrado: {VIDEO_IN}')
    cap = cv2.VideoCapture(VIDEO_IN)
    if not cap.isOpened():
        raise RuntimeError('Falha ao abrir stream de video.')

    fps = cap.get(cv2.CAP_PROP_FPS)
    largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    params = calcular_parametros_dinamicos(fps, largura, altura)
    zona = (int(ZONA_PROMO_REL[0]*largura), int(ZONA_PROMO_REL[1]*altura),
            int(ZONA_PROMO_REL[2]*largura), int(ZONA_PROMO_REL[3]*altura))

    writer = cv2.VideoWriter(TEMP_OUT, cv2.VideoWriter_fourcc(*'mp4v'), params.fps, (largura, altura))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError('Falha ao inicializar VideoWriter.')

    print('Parametros dinamicos:')
    print(f'  FPS={params.fps:.2f}, resolucao={largura}x{altura}, dt={params.dt:.4f}s')
    print(f'  oclusao={params.max_frames_oclusao} frames ({params.max_frames_oclusao/params.fps:.2f}s)')
    print(f'  dist_tracker={params.max_dist_tracker:.1f}px, dist_redeteccao={params.max_dist_redeteccao:.1f}px')
    print(f'  Q_base={params.process_noise:.3f}, R={params.measurement_noise:.3f}')

    kf = criar_kalman(params)
    trackers = {}
    estado = 'PERDIDO'
    origem_confirmacao = 'NENHUMA'
    frames_sem_consenso = 0
    trilha = deque(maxlen=max(30, int(3*params.fps)))
    hist_ref = None
    hist_adapt = None
    cx = cy = cw = ch = 0.0
    votos_validos = 0
    maioria_necessaria = 0
    modo_consenso = 'SEM_TRACKER'
    ancora_perda = (cx, cy)
    raio_busca_atual = params.max_dist_redeteccao
    sigma_kalman = 0.0

    metricas = {
        'total_frames': 0, 'confirmados': 0, 'previstos': 0, 'perdidos': 0,
        'confirmacoes_visuais': 0, 'inicializacoes_manuais': 0,
        'recuperacoes': 0, 'recuperacoes_baixa_confianca': 0,
        'falhas_redeteccao': 0, 'zona_confirmada': 0, 'zona_estimada': 0,
        'tempo_inicio': time.time()
    }

    lido, primeiro = cap.read()
    if not lido:
        cap.release(); writer.release()
        raise RuntimeError('Video vazio ou primeiro frame ilegivel.')

    bbox, origem_init, selecao_manual, candidatos_iniciais = selecionar_alvo_inicial(primeiro)
    visual_init = origem_init == 'HOG_INICIAL'
    if selecao_manual:
        metricas['inicializacoes_manuais'] += 1
        print('[AVISO] Selecao manual/fallback nao conta como confirmacao visual.')
        exibir_previa_bbox_manual(primeiro, bbox)
    elif len(candidatos_iniciais) > 1:
        print(f'[SELECAO] Alvo automatico escolhido pelo maior peso HOG entre {len(candidatos_iniciais)} pessoas.')

    cw, ch = float(bbox[2]), float(bbox[3])
    cx, cy = bbox[0]+cw/2, bbox[1]+ch/2
    inicializar_kalman(kf, cx, cy, params)
    hist_ref = estabelecer_assinatura_inicial(
        primeiro, bbox, origem_init, manual_confirmada=BBOX_MANUAL_CONFIRMADA
    )
    hist_adapt = hist_ref.copy() if hist_ref is not None else None
    if not identidade_manual_liberada(origem_init, BBOX_MANUAL_CONFIRMADA, hist_ref):
        cap.release()
        writer.release()
        if os.path.exists(TEMP_OUT):
            os.remove(TEMP_OUT)
        raise RuntimeError(
            'Inicializacao manual pendente. Ajuste BBOX_MANUAL apos conferir a previa, '
            'defina BBOX_MANUAL_CONFIRMADA=True e execute processar_video() novamente. '
            'Nenhum tracker foi iniciado e nenhum estado CONFIRMADO foi emitido.'
        )
    trackers = inicializar_trackers(primeiro, bbox, habilitar_goturn=goturn_ok)

    # Confirmacao requer evidencia visual e pelo menos um tracker inicializado.
    if visual_init and trackers:
        estado, origem_confirmacao = 'CONFIRMADO', origem_init
    else:
        estado, origem_confirmacao = 'PREVISTO', ('MANUAL' if not visual_init else 'NENHUMA')
    # O primeiro frame ja foi consumido e usado na inicializacao. Ele entra na
    # saida e nas metricas sem predict()/update(), pois nenhum dt transcorreu.
    primeiro_saida = primeiro.copy()
    cv2.rectangle(primeiro_saida, (zona[0], zona[1]),
                  (zona[0]+zona[2], zona[1]+zona[3]), (255,120,0), 2)
    cor_inicial = (0,255,0) if estado == 'CONFIRMADO' else (0,165,255)
    rotulo_inicial = ('INICIAL AUTOMATICO HOG' if visual_init else
                      ('INICIAL MANUAL SEM HSV confianca=REDUZIDA' if hist_ref is None
                       else 'INICIAL MANUAL CONFIRMADO'))
    cv2.rectangle(primeiro_saida, (int(cx-cw/2), int(cy-ch/2)),
                  (int(cx+cw/2), int(cy+ch/2)), cor_inicial, 2)
    cv2.putText(primeiro_saida, rotulo_inicial, (15,30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, cor_inicial, 2)
    contabilizar_estado(metricas, estado)
    na_zona_inicial = (zona[0] <= cx <= zona[0]+zona[2] and
                       zona[1] <= cy <= zona[1]+zona[3])
    if na_zona_inicial and estado == 'CONFIRMADO':
        metricas['zona_confirmada'] += 1
    elif na_zona_inicial and estado == 'PREVISTO':
        metricas['zona_estimada'] += 1
    trilha.append((int(cx), int(cy)))
    writer.write(primeiro_saida)

    # cap ja aponta para o segundo frame. Cada iteracao seguinte corresponde
    # exatamente a um predict() e a um avanco temporal real.
    while True:
        lido, frame = cap.read()
        if not lido:
            break
        pred = kf.predict()
        recuperado_neste_frame = False
        if estado == 'PERDIDO':
            kx, ky, raio_busca_atual, sigma_kalman = limitar_predicao_perdida(
                kf, pred, ancora_perda, params, largura, altura
            )
        else:
            kx, ky = float(pred[0, 0]), float(pred[1, 0])
            raio_busca_atual = params.max_dist_redeteccao
            sigma_kalman = 0.0
        caixas_validas = {}
        votos_validos = 0
        maioria_necessaria = len(trackers)//2 + 1 if trackers else 0

        estado_entrada = estado
        # Em PERDIDO, trackers antigos ficam impedidos de votar. A unica saida e HOG+HSV
        # (ou HOG geometrico de confianca reduzida quando nao existe assinatura).
        if estado_entrada != 'PERDIDO':
            for nome, tracker in list(trackers.items()):
                try:
                    ok, box = tracker.update(frame)
                except Exception as exc:
                    print(f'[FALHA UPDATE] {nome}: {type(exc).__name__}: {exc}')
                    continue
                if ok and box[2] > 0 and box[3] > 0:
                    tcx, tcy = box[0]+box[2]/2, box[1]+box[3]/2
                    if math.hypot(tcx-kx, tcy-ky) <= params.max_dist_tracker:
                        caixas_validas[nome] = tuple(float(v) for v in box)
            box_consenso, votos_validos, maioria_necessaria, modo_consenso = obter_consenso_ensemble(
                caixas_validas, len(trackers)
            )
            houve_consenso = box_consenso is not None
            estado, frames_sem_consenso = decidir_estado(
                estado, houve_consenso, frames_sem_consenso, params.max_frames_oclusao
            )
            if estado == 'PERDIDO':
                ancora_perda = (cx, cy)
                # Zera deslocamento agressivo; a incerteza, nao uma trajetoria ilimitada,
                # passa a ampliar a busca HOG.
                kf.statePost[2, 0] *= KALMAN_VEL_DAMPING_PERDIDO
                kf.statePost[3, 0] *= KALMAN_VEL_DAMPING_PERDIDO
                trackers = {}  # invalida definitivamente os trackers antigos
                votos_validos = 0
                maioria_necessaria = 0
                modo_consenso = 'SEM_TRACKER'
        else:
            box_consenso = None
            houve_consenso = False
            trackers = {}
            votos_validos = 0
            maioria_necessaria = 0
            modo_consenso = 'SEM_TRACKER'

        if houve_consenso:
            cx = box_consenso[0]+box_consenso[2]/2
            cy = box_consenso[1]+box_consenso[3]/2
            cw, ch = box_consenso[2], box_consenso[3]
            kf.correct(np.array([[cx], [cy]], np.float32))
            origem_confirmacao = 'ENSEMBLE'
            metricas['confirmacoes_visuais'] += 1
            if metricas['total_frames'] % params.hsv_update_interval_frames == 0:
                hist_adapt, atualizou, dref = atualizar_assinatura_controlada(
                    frame, box_consenso, hist_ref, hist_adapt
                )
                if not atualizou and dref is not None:
                    print(f'[HSV] Atualizacao adaptativa rejeitada (dist_ref={dref:.3f}).')
        elif estado == 'PREVISTO':
            # Sem correct(): promove statePre/errorCovPre para que o proximo frame
            # parta desta previsao e acumule incerteza.
            propagar_sem_medicao(kf)
            cx, cy = float(kf.statePost[0, 0]), float(kf.statePost[1, 0])
            origem_confirmacao = 'KALMAN'
        else:
            origem_confirmacao = 'NENHUMA'
            if estado_entrada != 'PERDIDO':
                kx, ky, raio_busca_atual, sigma_kalman = limitar_predicao_perdida(
                    kf, pred, ancora_perda, params, largura, altura
                )
            resultado = tentar_redeteccao(
                frame, kx, ky, raio_busca_atual,
                area_esperada=max(1.0, cw*ch), hist_ref=hist_ref, hist_adapt=hist_adapt
            )
            if resultado.bbox is not None:
                novos_trackers = inicializar_trackers(frame, resultado.bbox, habilitar_goturn=goturn_ok)
                # Recuperacao so termina com deteccao compativel + ao menos um tracker iniciado.
                if novos_trackers:
                    trackers = novos_trackers
                    box = resultado.bbox
                    cw, ch = float(box[2]), float(box[3])
                    cx, cy = box[0]+cw/2, box[1]+ch/2
                    inicializar_kalman(kf, cx, cy, params)
                    estado = 'CONFIRMADO'
                    recuperado_neste_frame = True
                    origem_confirmacao = resultado.origem
                    frames_sem_consenso = 0
                    votos_validos = 0  # ainda nao houve update/voto no frame de reinicializacao
                    maioria_necessaria = len(trackers)//2 + 1
                    modo_consenso = 'TRACKER_UNICO' if len(trackers) == 1 else 'ENSEMBLE_REINICIALIZADO'
                    metricas['recuperacoes'] += 1
                    metricas['confirmacoes_visuais'] += 1
                    if resultado.confianca == 'REDUZIDA':
                        metricas['recuperacoes_baixa_confianca'] += 1
                    # Nunca substitui hist_ref. Atualiza adaptativa somente se compativel.
                    hist_adapt, _, _ = atualizar_assinatura_controlada(
                        frame, box, hist_ref, hist_adapt
                    )
                else:
                    metricas['falhas_redeteccao'] += 1
                    estado = 'PERDIDO'
                    origem_confirmacao = 'NENHUMA'
            else:
                metricas['falhas_redeteccao'] += 1

        contabilizar_estado(metricas, estado)
        validar_metricas(metricas)

        na_zona = (zona[0] <= cx <= zona[0]+zona[2] and zona[1] <= cy <= zona[1]+zona[3])
        if na_zona and estado == 'CONFIRMADO':
            metricas['zona_confirmada'] += 1
        elif na_zona and estado == 'PREVISTO':
            metricas['zona_estimada'] += 1

        if estado == 'PERDIDO':
            if not trilha or trilha[-1] is not None:
                trilha.append(None)
        else:
            trilha.append((int(cx), int(cy)))

        # HUD fiel ao estado, votos validos e trackers ativos.
        cv2.rectangle(frame, (zona[0], zona[1]), (zona[0]+zona[2], zona[1]+zona[3]), (255,120,0), 2)
        if recuperado_neste_frame or estado == 'CONFIRMADO':
            cor = (0,255,0)
        elif estado == 'PREVISTO':
            cor = (0,165,255)
        else:
            cor = (0,0,255)
        txt = construir_texto_hud(
            estado, origem_confirmacao, modo_consenso, votos_validos, len(trackers),
            maioria_necessaria, frames_sem_consenso, params.max_frames_oclusao,
            hist_ref is not None, recuperado_neste_frame,
            raio_busca_atual if estado == 'PERDIDO' else None,
            sigma_kalman if estado == 'PERDIDO' else None
        )

        if estado != 'PERDIDO':
            x1, y1 = int(cx-cw/2), int(cy-ch/2)
            x2, y2 = int(cx+cw/2), int(cy+ch/2)
            cv2.rectangle(frame, (x1,y1), (x2,y2), cor, 2)
        for i in range(1, len(trilha)):
            if trilha[i-1] is not None and trilha[i] is not None:
                cv2.line(frame, trilha[i-1], trilha[i], (0,255,255), 2)
        cv2.putText(frame, txt, (15,30), cv2.FONT_HERSHEY_SIMPLEX, 0.55, cor, 2)
        writer.write(frame)

    cap.release()
    writer.release()
    metricas['tempo_total'] = time.time() - metricas['tempo_inicio']
    validar_metricas(metricas)

    print('\nProcessamento concluido. Iniciando encoding H.264...')
    proc = subprocess.run(
        ['ffmpeg', '-y', '-i', TEMP_OUT, '-vcodec', 'libx264', VIDEO_OUT, '-loglevel', 'error'],
        capture_output=True, text=True
    )
    if proc.returncode != 0 or not os.path.exists(VIDEO_OUT):
        print(f'[ERRO FFMPEG] retcode={proc.returncode}: {proc.stderr.strip()}')
        return metricas
    if os.path.exists(TEMP_OUT):
        os.remove(TEMP_OUT)

    total = metricas['total_frames']
    print('='*52)
    print('ESTATISTICAS DE RASTREAMENTO')
    print('='*52)
    print(f'Total de frames            : {total}')
    print(f'CONFIRMADO                  : {metricas["confirmados"]}')
    print(f'PREVISTO                    : {metricas["previstos"]}')
    print(f'PERDIDO                     : {metricas["perdidos"]}')
    print(f'Soma das categorias         : {metricas["confirmados"] + metricas["previstos"] + metricas["perdidos"]}')
    print(f'Confirmacoes visuais        : {metricas["confirmacoes_visuais"]}')
    print(f'Inicializacoes manuais      : {metricas["inicializacoes_manuais"]}')
    print(f'Recuperacoes                : {metricas["recuperacoes"]}')
    print(f'Recuperacoes conf. reduzida : {metricas["recuperacoes_baixa_confianca"]}')
    print(f'Falhas de redeteccao        : {metricas["falhas_redeteccao"]}')
    print(f'Tempo na zona confirmado    : {metricas["zona_confirmada"]/params.fps:.1f}s')
    print(f'Tempo na zona previsto      : {metricas["zona_estimada"]/params.fps:.1f}s')
    print(f'Tempo de processamento      : {metricas["tempo_total"]:.1f}s')
    print('='*52)

    with open(VIDEO_OUT, 'rb') as arquivo:
        video64 = base64.b64encode(arquivo.read()).decode()
    display(HTML(f'<video width="800" controls><source src="data:video/mp4;base64,{video64}" type="video/mp4"></video>'))
    return metricas


# ------------------------- 8. TESTES CONTROLADOS -------------------------
class TrackerMock:
    def __init__(self, respostas):
        self.respostas = iter(respostas)
        self.chamadas = 0
    def update(self, frame):
        self.chamadas += 1
        return next(self.respostas)


class TestesEstados(unittest.TestCase):
    def test_consenso_exige_todos_os_pares(self):
        # A concorda com B e C; B e C discordam. Nunca pode haver tres votos.
        caixas = {'A': (30,0,100,100), 'B': (0,0,100,100), 'C': (60,0,100,100)}
        self.assertGreaterEqual(calcular_iou(caixas['A'], caixas['B']), 0.40)
        self.assertGreaterEqual(calcular_iou(caixas['A'], caixas['C']), 0.40)
        self.assertLess(calcular_iou(caixas['B'], caixas['C']), 0.40)
        box, votos, maioria, modo = obter_consenso_ensemble(caixas, 3)
        self.assertIsNotNone(box)
        self.assertEqual(votos, 2)
        self.assertEqual(modo, 'ENSEMBLE')

    def test_rejeita_geometria_divergente_sem_maioria(self):
        caixas = {'A': (0,0,20,20), 'B': (100,100,20,20), 'C': (200,200,20,20)}
        box, votos, maioria, _ = obter_consenso_ensemble(caixas, 3)
        self.assertIsNone(box); self.assertEqual(votos, 0); self.assertEqual(maioria, 2)

    def test_tracker_unico_modo_degradado(self):
        box, votos, maioria, modo = obter_consenso_ensemble({'KCF': (1,2,30,40)}, 1)
        self.assertIsNotNone(box); self.assertEqual(votos, 1); self.assertEqual(maioria, 1)
        self.assertEqual(modo, 'TRACKER_UNICO')

    def test_transicoes_confirmado_previsto_perdido(self):
        estado, n = decidir_estado('CONFIRMADO', False, 0, 1)
        self.assertEqual((estado,n), ('PREVISTO',1))
        estado, n = decidir_estado(estado, False, n, 1)
        self.assertEqual(estado, 'PERDIDO')

    def test_bloqueio_tracker_antigo_e_recuperacao_validada(self):
        antigo = TrackerMock([(True, (0,0,20,20))])
        estado = 'PERDIDO'
        # Replica a regra real do loop: PERDIDO nao chama update nem aceita voto antigo.
        caixas = {}
        if estado != 'PERDIDO':
            ok, box = antigo.update(None)
            if ok: caixas['ANTIGO'] = box
        self.assertEqual(antigo.chamadas, 0)
        self.assertFalse(caixas)
        hog_ok, hsv_ok, novos = True, False, {'NOVO': object()}
        if hog_ok and hsv_ok and novos: estado = 'CONFIRMADO'
        self.assertEqual(estado, 'PERDIDO')
        hsv_ok = True
        if hog_ok and hsv_ok and novos: estado = 'CONFIRMADO'
        self.assertEqual(estado, 'CONFIRMADO')

    def test_assinatura_original_nao_e_substituida(self):
        frame_ref = np.zeros((40,40,3), np.uint8); frame_ref[:] = (0,0,255)
        frame_outro = np.zeros((40,40,3), np.uint8); frame_outro[:] = (255,0,0)
        ref = extrair_histograma(frame_ref, (0,0,40,40)); adapt = ref.copy(); ref_copia = ref.copy()
        novo, atualizou, _ = atualizar_assinatura_controlada(frame_outro, (0,0,40,40), ref, adapt)
        self.assertFalse(atualizou)
        self.assertTrue(np.array_equal(ref, ref_copia))
        self.assertTrue(np.array_equal(novo, adapt))

    def test_continuidade_mock_apos_reinicializacao(self):
        novo = TrackerMock([(True, (10,10,20,20)), (True, (11,10,20,20))])
        caixas1, caixas2 = {}, {}
        for destino in (caixas1, caixas2):
            ok, box = novo.update(None)
            if ok: destino['NOVO'] = box
        self.assertEqual(novo.chamadas, 2)
        self.assertIsNotNone(obter_consenso_ensemble(caixas1,1)[0])
        self.assertIsNotNone(obter_consenso_ensemble(caixas2,1)[0])

    def test_propagacao_sem_medicao_avanca_posicao(self):
        params = calcular_parametros_dinamicos(10, 640, 480)
        kf = criar_kalman(params)
        inicializar_kalman(kf, 100, 100, params)
        kf.statePost[2, 0] = 20.0
        xs = []
        for _ in range(5):
            kf.predict()
            propagar_sem_medicao(kf)
            xs.append(float(kf.statePost[0, 0]))
        self.assertTrue(all(b > a for a, b in zip(xs, xs[1:])))
        self.assertAlmostEqual(xs[-1] - xs[0], 4 * 20 * params.dt, places=3)
        self.assertAlmostEqual(float(kf.statePost[2, 0]), 20.0, places=4)

    def test_covariancia_acumula_sem_medicao(self):
        params = calcular_parametros_dinamicos(25, 1280, 720)
        kf = criar_kalman(params)
        inicializar_kalman(kf, 200, 200, params)
        traces = []
        for _ in range(12):
            kf.predict(); propagar_sem_medicao(kf)
            traces.append(float(np.trace(kf.errorCovPost[:2, :2])))
        self.assertTrue(all(b > a for a, b in zip(traces, traces[1:])), traces)

    def test_raio_busca_cresce_efetivamente(self):
        params = calcular_parametros_dinamicos(30, 1920, 1080)
        kf = criar_kalman(params); inicializar_kalman(kf, 960, 540, params)
        ancora = (960.0, 540.0); raios = []
        for _ in range(20):
            pred = kf.predict()
            _, _, raio, _ = limitar_predicao_perdida(kf, pred, ancora, params, 1920, 1080)
            raios.append(raio)
        self.assertGreater(raios[-1], raios[0])
        self.assertTrue(any(b > a for a, b in zip(raios, raios[1:])))

    def test_manual_pendente_bloqueia_inicio(self):
        frame = np.zeros((50, 50, 3), np.uint8); frame[:] = (0, 0, 255)
        hist = estabelecer_assinatura_inicial(frame, (0,0,50,50), 'MANUAL', False)
        self.assertFalse(identidade_manual_liberada('MANUAL', False, hist))

    def test_manual_confirmada_cria_assinatura(self):
        frame = np.zeros((50, 50, 3), np.uint8); frame[:] = (0, 0, 255)
        hist = estabelecer_assinatura_inicial(frame, (0,0,50,50), 'MANUAL', True)
        self.assertIsNotNone(hist)
        self.assertTrue(identidade_manual_liberada('MANUAL', True, hist))

    def test_kalman_perdido_limita_deriva_e_amplia_busca(self):
        params = calcular_parametros_dinamicos(30, 1920, 1080)
        kf = criar_kalman(params)
        inicializar_kalman(kf, 960, 540, params)
        kf.statePost[2, 0], kf.statePost[3, 0] = 100000, 100000
        ancora = (960.0, 540.0)
        raios = []
        for _ in range(30):
            pred = kf.predict()
            x, y, raio, sigma = limitar_predicao_perdida(kf, pred, ancora, params, 1920, 1080)
            raios.append(raio)
            self.assertLessEqual(math.hypot(x-ancora[0], y-ancora[1]),
                                 KALMAN_MAX_DRIFT_DIAG_REL*params.diagonal + 1e-3)
            self.assertLessEqual(math.hypot(float(kf.statePost[2,0]), float(kf.statePost[3,0])),
                                 KALMAN_VEL_MAX_DIAG_S*params.diagonal + 1e-3)
        self.assertGreaterEqual(raios[-1], raios[0])
        self.assertLessEqual(raios[-1], DIST_REDETECCAO_MAX_DIAG_REL*params.diagonal)

    def test_bbox_manual_nao_confirmada_nao_cria_identidade(self):
        frame = np.zeros((80,80,3), np.uint8)
        frame[:, :] = (20,20,20)
        hist = estabelecer_assinatura_inicial(frame, (0,0,40,40), 'MANUAL', False)
        self.assertIsNone(hist)
        hist_confirmado = estabelecer_assinatura_inicial(frame, (0,0,40,40), 'MANUAL', True)
        # Uma ROI uniforme/sem saturacao tambem nao produz assinatura util.
        self.assertIsNone(hist_confirmado)
        self.assertTrue(identidade_manual_liberada('MANUAL', True, hist_confirmado))

    def test_hud_recuperacao_nao_finge_consenso(self):
        txt = construir_texto_hud('CONFIRMADO', 'HOG_HSV', 'ENSEMBLE_REINICIALIZADO',
                                  0, 3, 2, 0, 15, True, True)
        self.assertIn('RECUPERADO HOG_HSV', txt)
        self.assertIn('aguardando votos', txt)
        self.assertNotIn('votos=0/3', txt)
        proximo = construir_texto_hud('CONFIRMADO', 'ENSEMBLE', 'ENSEMBLE',
                                      2, 3, 2, 0, 15, True, False)
        self.assertIn('CONFIRMADO ENSEMBLE', proximo)
        self.assertIn('votos=2/3', proximo)

    def test_recuperacao_apos_perda_prolongada(self):
        params = calcular_parametros_dinamicos(20, 640, 480)
        kf = criar_kalman(params); inicializar_kalman(kf, 320, 240, params)
        ancora = (320.0, 240.0)
        for _ in range(40):
            pred = kf.predict()
            limitar_predicao_perdida(kf, pred, ancora, params, 640, 480)
        # Recuperacao visual reinicializa estado e permite continuidade do mock.
        inicializar_kalman(kf, 350, 250, params)
        novo = TrackerMock([(True, (340,240,20,20)), (True, (342,240,20,20))])
        for _ in range(2):
            ok, box = novo.update(None)
            self.assertTrue(ok)
            consenso = obter_consenso_ensemble({'NOVO': box}, 1)[0]
            self.assertIsNotNone(consenso)
        self.assertEqual(novo.chamadas, 2)

    def test_reinicializacao_kalman_limpa_covariancia(self):
        params = calcular_parametros_dinamicos(30, 1280, 720)
        kf = criar_kalman(params)
        inicializar_kalman(kf, 100, 100, params)
        for _ in range(50):
            kf.predict(); propagar_sem_medicao(kf)
        cov_alta = float(np.trace(kf.errorCovPost))
        inicializar_kalman(kf, 300, 250, params)
        cov_reset = float(np.trace(kf.errorCovPost))
        self.assertLess(cov_reset, cov_alta)
        self.assertTrue(np.allclose(kf.errorCovPost, kf.errorCovPre))
        self.assertAlmostEqual(float(kf.statePost[0,0]), 300.0)
        self.assertAlmostEqual(float(kf.statePost[2,0]), 0.0)

    def test_manual_confirmada_sem_hsv_libera_modo_geometrico(self):
        self.assertTrue(identidade_manual_liberada('MANUAL', True, None))
        self.assertFalse(identidade_manual_liberada('MANUAL', False, None))
        resultado = construir_texto_hud('PERDIDO', 'NENHUMA', 'SEM_TRACKER',
                                        0, 0, 0, 20, 15, False,
                                        recuperado_neste_frame=False,
                                        raio_busca=200, sigma=10)
        self.assertIn('HOG geometria', resultado)

    def test_alinhamento_frames_e_passos_kalman(self):
        for total in [0, 1, 2, 10, 101]:
            self.assertEqual(passos_previsao_esperados(total), max(0, total-1))
        # Primeiro frame e escrito/contado sem previsao; demais geram um passo.
        total = 7
        frames_escritos = 1
        passos = 0
        for _ in range(1, total):
            passos += 1
            frames_escritos += 1
        self.assertEqual(frames_escritos, total)
        self.assertEqual(passos, passos_previsao_esperados(total))

    def test_metricas_exclusivas(self):
        m = {'total_frames':0, 'confirmados':0, 'previstos':0, 'perdidos':0}
        for e in ['CONFIRMADO','PREVISTO','PERDIDO','CONFIRMADO']:
            contabilizar_estado(m,e)
        validar_metricas(m); self.assertEqual(m['total_frames'],4)

    def test_fluxo_controlado_integrado(self):
        # Tracker mock executa o fluxo real de update -> consenso -> estados -> perda.
        t = TrackerMock([(True,(0,0,20,20)), (False,(0,0,0,0)), (False,(0,0,0,0))])
        estado, sem = 'CONFIRMADO', 0
        for esperado in ['CONFIRMADO','PREVISTO','PERDIDO']:
            caixas={}
            if estado != 'PERDIDO':
                ok, box=t.update(None)
                if ok: caixas['T']=box
            consenso=obter_consenso_ensemble(caixas,1)[0] is not None
            estado, sem=decidir_estado(estado,consenso,sem,1)
            self.assertEqual(estado,esperado)
        self.assertEqual(t.chamadas,3)

def executar_testes():
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestesEstados)
    resultado = unittest.TextTestRunner(verbosity=2).run(suite)
    if not resultado.wasSuccessful():
        raise AssertionError('Os testes controlados falharam.')
    return resultado


if __name__ == '__main__':
    parser = argparse.ArgumentParser(add_help=True)
    parser.add_argument('--self-test', action='store_true', help='Executa apenas testes controlados.')
    parser.add_argument('--processar', action='store_true', help='Processa VIDEO_IN.')
    args, _ = parser.parse_known_args()
    if args.self_test:
        executar_testes()
    elif args.processar:
        processar_video()
    else:
        print('Modulo carregado. Use executar_testes() ou processar_video() no Colab.')


Modulo carregado. Use executar_testes() ou processar_video() no Colab.


In [20]:
executar_testes()

test_alinhamento_frames_e_passos_kalman (__main__.TestesEstados.test_alinhamento_frames_e_passos_kalman) ... ok
test_assinatura_original_nao_e_substituida (__main__.TestesEstados.test_assinatura_original_nao_e_substituida) ... ok
test_bbox_manual_nao_confirmada_nao_cria_identidade (__main__.TestesEstados.test_bbox_manual_nao_confirmada_nao_cria_identidade) ... ok
test_bloqueio_tracker_antigo_e_recuperacao_validada (__main__.TestesEstados.test_bloqueio_tracker_antigo_e_recuperacao_validada) ... ok
test_consenso_exige_todos_os_pares (__main__.TestesEstados.test_consenso_exige_todos_os_pares) ... ok
test_continuidade_mock_apos_reinicializacao (__main__.TestesEstados.test_continuidade_mock_apos_reinicializacao) ... ok
test_covariancia_acumula_sem_medicao (__main__.TestesEstados.test_covariancia_acumula_sem_medicao) ... ok
test_fluxo_controlado_integrado (__main__.TestesEstados.test_fluxo_controlado_integrado) ... ok
test_hud_recuperacao_nao_finge_consenso (__main__.TestesEstados.test_hud_r

[HSV] BBOX_MANUAL ainda nao confirmada; histograma de referencia nao foi criado.
[HSV] BBOX_MANUAL ainda nao confirmada; histograma de referencia nao foi criado.


<unittest.runner.TextTestResult run=21 errors=0 failures=0>

In [21]:
processar_video()

Modelo GOTURN disponivel.
Parametros dinamicos:
  FPS=24.00, resolucao=1280x720, dt=0.0417s
  oclusao=14 frames (0.58s)
  dist_tracker=176.2px, dist_redeteccao=367.2px
  Q_base=13.480, R=19.411
[SELECAO] Alvo automatico escolhido pelo maior peso HOG entre 5 pessoas.
[INDISPONIVEL] GOTURN: GOTURN nao inicializado. cv2: error: OpenCV(4.14.0) /io/opencv/modules/video/src/tracking/tracker_goturn.cpp:44: error: (-215:Assertion failed) !net.empty() in function 'TrackerGOTURNImpl'
 | cv2.legacy: fabrica ausente
[FALHA INIT] CSRT: error: OpenCV(4.14.0) :-1: error: (-5:Bad argument) in function 'init'
> Overload resolution failed:
>  - Can't parse 'boundingBox'. Sequence item with index 0 has a wrong type
>  - Can't parse 'boundingBox'. Sequence item with index 0 has a wrong type

[FALHA INIT] KCF: error: OpenCV(4.14.0) :-1: error: (-5:Bad argument) in function 'init'
> Overload resolution failed:
>  - Can't parse 'boundingBox'. Sequence item with index 0 has a wrong type
>  - Can't parse 'boun

{'total_frames': 240,
 'confirmados': 0,
 'previstos': 15,
 'perdidos': 225,
 'confirmacoes_visuais': 0,
 'inicializacoes_manuais': 0,
 'recuperacoes': 0,
 'recuperacoes_baixa_confianca': 0,
 'falhas_redeteccao': 225,
 'zona_confirmada': 0,
 'zona_estimada': 15,
 'tempo_inicio': 1793658425.240933,
 'tempo_total': 208.4128875732422}